In [1]:
%load_ext autoreload
%autoreload 2

# Compute coupling for DRN models

We will utilize Enesmble Corpula Coupling (ECC) and a Gaussian Corpula Approach (GCA)

## ECC

Steps:
1. **Univariate postprocessing:** Obtain univariately postprocessed marginal distibutions for each margin. Here we use location and variable as the margins. This is simply the ouput of a model that inherits from `DistibutionRegression`.

2. **Quantization:** Draw samples from each marginal predictive distibution. The samples should have the same size as the raw ensemble $M$. We sample at equidistant quantiles $\frac{1}{M+1}, \dots, \frac{M}{M+1}$. Note that for the wind speed the distribution is a truncated normal. This yields the samples $\tilde{x}^l_1, \dots, \tilde{x}^l_M$, where $l \in L=I \times J$, $I$ is the set of stations and $J$ is the set of features (variables).

3. **Ensemble Reordering:** Reorder based on the rank structure of the original ensemble with possible ties resolved at random. For each margin $l$, the order statistics of the raw ensemble values $x_{(1)}^l, \dots, x_{(M)}^l$ introduce a permutation $\sigma_l$ of the integers ${1, \dots, M}$ defined by $\sigma_l(m)=\operatorname{rk}(x_m^l)$ for $m = 1, \dots, M$. Ties in the rank are resolved at random. The respecitve margin of the ECC postprocessed ensemble is then given by $\hat{x}=\tilde{x}_{\sigma_l(1)}, \dots, \tilde{x}_{\sigma_l(M)}$.

## GCA
First of all some problems with this approach:
- In the common case in which the dimension L (= all station, variable, lead time pairs) of the output quantity is huge and no specific structure can be exploited, parametric methods are bound to fail.

Steps:
1. **Transform to latent Gaussian:** Transform a set of past observarions into latent standard Gaussian observations via $$\tilde{y}^l=\Phi^{-1}(F_\theta^l(y^l))$$ where $F_\theta^d$ is the distribution for each margin.
2. **Sampling:** Calculate $\Sigma$ based on $\tilde{y}^l$ obtained in Step 1. Then sample from $N(0,\Sigma)$ to obtain $Z_1, \dots, Z_M$.
3. **Reverse Transform:** Transform back to original space: $$X_m^l=(F_\theta^l)^{-1}(\Phi(Z_m^l))$$



In [ ]:
import importlib
from collections import defaultdict

import hydra
import lightning as L
import numpy as np
import torch
import xarray as xr
from einops import rearrange, reduce
from omegaconf import DictConfig
from scipy.stats import multivariate_normal, norm, truncnorm
from tqdm import tqdm

from genpp import BASE_DIR
from genpp.configs import register_resolvers
from genpp.data import FC_VARS, FORECAST_ENS_PATH, OBSERVATIONS_FLAT_PATH
from genpp.eval.utils import log_scores, update_wandb_run
from genpp.models.loss import EnergyScore, EnsembleCRPS, VariogramScore

try:
    register_resolvers()
except Exception:
    pass

best emos model: `feik/genpp/k32mygar`

best drn model: `feik/genpp/hn0gdrqm`

In [3]:
RUN_PATH = "feik/genpp/k32mygar"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

SCORE_FILE = MODEL_DIR / "scores.csv"

In [4]:
# Load model and dataloader to get predictions
with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg: DictConfig = hydra.compose(
        config_name="config",
    )
    # Do not shuffle any dataloader
    cfg.data.module.dataloader_config.train.shuffle = False
    cfg.data.module.dataloader_config.val.shuffle = False
    cfg.data.module.dataloader_config.test.shuffle = False
    datamodule = hydra.utils.instantiate(cfg.data.module)
    model_class = cfg.model._target_.split(".")[-1]
datamodule.prepare_data()
datamodule.setup(stage="validate")

Configuration hash: 5c2f4f1833d7163c987092b088f1dd92574122da72aa29df0e41d7cb02738f08
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_5c2f4f1833d7163c987092b088f1dd92574122da72aa29df0e41d7cb02738f08.pt.


In [5]:
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

model = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

In [6]:
trainer = L.Trainer(logger=False)
predictions = trainer.predict(model, datamodule.val_dataloader(), return_predictions=True)

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A5000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1

Predicting: |          | 0/? [00:00<?, ?it/s]

Predicting DataLoader 0: 100%|██████████| 57/57 [00:00<00:00, 124.58it/s]


In [7]:
def stack_predictions(predictions) -> list[dict[str, torch.Tensor]]:
    n_dicts = len(predictions[0])
    merged = [defaultdict(list) for _ in range(n_dicts)]

    # accumulate tensors per key
    for batch in predictions:
        for i, d in enumerate(batch):
            for k, v in d.items():
                merged[i][k].append(v)

    # concatenate along first dimension
    return [{k: torch.cat(vs, dim=0) for k, vs in m.items()} for m in merged]


def predictions_to_dataarray(obs_da, preds):
    """
    obs_da: original xarray.DataArray of observations
    preds: list of dicts, each dict corresponds to one feature
           dict values are torch.Tensors with shape (time, longitude, latitude)
    """
    features = obs_da.coords["feature"].values  # ["2m_temperature", "10m_wind_speed"]

    new_data = []
    new_feature_names = []

    for feat_name, pred_dict in zip(features, preds):
        for param_name, tensor in pred_dict.items():
            # ensure numpy
            arr = tensor.detach().cpu().numpy()
            new_data.append(arr)
            new_feature_names.append(f"{feat_name}_{param_name}")

    # stack along new "feature" axis
    data = np.stack(new_data, axis=1)  # (time, new_features, longitude, latitude)
    print(data.shape)

    # build DataArray
    return xr.DataArray(
        data,
        coords={
            "prediction": obs_da.coords["prediction"],
            "feature": new_feature_names,
            "longitude": obs_da.coords["longitude"],
            "latitude": obs_da.coords["latitude"],
            "prediction_time": ("prediction", obs_da.coords["prediction_time"].values),
        },
        dims=("prediction", "feature", "longitude", "latitude"),
    )

In [ ]:
# Next step get this in order to a nice datastructure
# To ensure the axes are labled correctly we use the train dataset to get the axis labels

import pickle

from genpp.data import FORECAST_ENS_FLAT_AGG_PATH

stacked = stack_predictions(predictions)
val_split = hydra.utils.instantiate(cfg.data.splits.val)


x_ds = xr.open_dataset(FORECAST_ENS_FLAT_AGG_PATH)

time_slice = {
    "train": hydra.utils.instantiate(cfg.data.splits.train),
    "val": hydra.utils.instantiate(cfg.data.splits.val),
    "test": hydra.utils.instantiate(cfg.data.splits.test),
}
val_slice = time_slice["val"]

x_ds = x_ds.stack(prediction=("time", "prediction_timedelta"))

# Lets investigate the len of the dataset splits
with open(datamodule.metadata_path, "rb") as f:
    cache_metadata = pickle.load(f)

train_idx = cache_metadata["split_indices"]["train"]
val_idx = cache_metadata["split_indices"]["val"]
test_idx = cache_metadata["split_indices"]["test"]

# Now lets get the actual times we need for the val set and and get the val data
val_prediction = x_ds.isel(prediction=val_idx).prediction
val_times = val_prediction.time + val_prediction.prediction_timedelta
val_times = val_times.compute().values

# Now load the y_val data for the val times
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=val_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", val_prediction.to_index()),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_da = y_val.sel(feature=feature_order)

# TODO this needs a rework with an time and timedelta axis instead of only time
predictions_xr = predictions_to_dataarray(y_da, stacked)

(3620, 4, 37, 31)


## ECC Step 2: Quantization
Draw equally spaced samples from the distributions where for the temperature the distibution is $N(\mu, \sigma)$.
And for the wind speed it is a truncated normal with bounds at 0 and $\infty$.
Since the ensemble has 50 members we will draw 50 samples.

In [9]:
def quantile_samples(pred_da, M=50):
    """
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']
    M: number of quantile samples

    Returns: xarray.DataArray with dims (time, feature, samples, longitude, latitude)
    """
    prediction = pred_da.coords["prediction"]
    longitude = pred_da.coords["longitude"]
    latitude = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # quantile positions
    qs = np.linspace(1 / (M + 1), M / (M + 1), M)

    # temperature (normal distribution)
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values  # (time, lon, lat)
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    temp_samples = norm.ppf(
        qs[:, None, None, None], loc=mu_temp[None, ...], scale=sigma_temp[None, ...]
    )
    # (M, time, lon, lat)

    # wind speed (truncated normal, bounds [0, ∞))
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    # From the Docs:
    # If a_trunc and b_trunc are the abscissae at which we wish to truncate the distribution
    # (as opposed to the number of standard deviations from loc),
    # then we can calculate the distribution parameters a and b as follows:
    # a, b = (a_trunc - loc) / scale, (b_trunc - loc) / scale

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    wind_samples = truncnorm.ppf(
        qs[:, None, None, None],
        a[None, ...],
        b,
        loc=mu_wind[None, ...],
        scale=sigma_wind[None, ...],
    )
    # (M, time, lon, lat)

    # stack along feature axis
    data = np.stack([temp_samples, wind_samples], axis=1)  # (M, feature, time, lon, lat)

    # reorder to (time, feature, sample, lon, lat)
    data = rearrange(data, "sample feature prediction lon lat -> prediction feature sample lon lat")

    return xr.DataArray(
        data,
        coords={
            "prediction": prediction,
            "feature": features,
            "sample": np.arange(M),
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("prediction", "feature", "sample", "longitude", "latitude"),
    )

In [10]:
pred_samples = quantile_samples(predictions_xr, M=50)

## ECC Step 3: Ensemble Reordering

Now load the Ensemble predictions and determine the rank ordering and use this to reshuffle the predicted samples

In [21]:
ens = (
    xr.open_dataset(FORECAST_ENS_PATH)[FC_VARS]
    .stack(prediction=("time", "prediction_timedelta"))
    .sel(prediction=val_prediction)
    .to_dataarray("feature")
    .transpose("prediction", "feature", "number", "longitude", "latitude")
)

In [34]:
# Random tie breaking is done by noising the data slightly to avoid ties
# Theoretically ties would still be possible, but insanely unlikely
rng = np.random.default_rng(seed=42)
noise = rng.uniform(low=-1e-10, high=1e-10, size=ens.shape)
ens_noised = ens + noise

ens_ranked = ens_noised.rank(dim="number") - 1  # -1 to get ranks from 0 to 49
ens_ranked = ens_ranked.astype(np.int32)

In [36]:
# Get the same dims so that the reodering works
indexer = ens_ranked
indexer = indexer.rename({"number": "sample"})
# Drop this var since for every sample we want the indexing array
# Note to me: why would we drop "sample" here?
indexer = indexer.drop_vars("sample", errors="ignore")

# Use advanced indexing
# Note to me: I tested this and it works correctly
pred_samples_reordered = pred_samples.isel(sample=indexer)

## Now compute the energy scores
For this we need to convert the underlying arrays to tensors and collect the scores afterwards
We can loop over the (batched) days to calculate these scores

In [43]:
SKIP_VARIOGRAM = True

In [40]:
crps = EnsembleCRPS()
es = EnergyScore(clamp=False)
vs = VariogramScore(p=0.5)

In [ ]:
pred_samples_reordered = pred_samples_reordered.transpose(
    "prediction", "sample", "feature", "longitude", "latitude"
)

prediction = y_da.prediction
crpss_list = []
ess_per_var_list = []
ess_complete_list = []
if not SKIP_VARIOGRAM:
    vss_per_var_list = []
    vss_complete_list = []
for t in tqdm(prediction):
    curr_obs = y_da.sel(prediction=t).to_numpy()
    curr_pred = pred_samples_reordered.sel(prediction=t).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)

    obs_feature = rearrange(obs_t, "b f lon lat -> b f (lon lat)")
    pred_feature = rearrange(pred_t, "b n f lon lat -> b f n (lon lat)")

    obs_complete = rearrange(obs_t, "b f lon lat -> b (f lon lat)")
    pred_complete = rearrange(pred_t, "b n f lon lat -> b n (f lon lat)")

    crps_per_margin = crps(pred_t, obs_t)
    es_per_var = es(pred_feature, obs_feature)
    es_complete = es(pred_complete, obs_complete)
    if not SKIP_VARIOGRAM:
        vs_per_var = vs(pred_feature, obs_feature)
        vs_complete = vs(pred_complete, obs_complete)

    crpss_list.append(crps_per_margin)
    ess_per_var_list.append(es_per_var)
    ess_complete_list.append(es_complete)
    if not SKIP_VARIOGRAM:
        vss_per_var_list.append(vs_per_var)  # type: ignore
        vss_complete_list.append(vs_complete)  # type: ignore

crpss = torch.cat(crpss_list, dim=0)
ess_per_var = torch.cat(ess_per_var_list, dim=0)
ess_complete = torch.cat(ess_complete_list, dim=0)
if not SKIP_VARIOGRAM:
    vss_per_var = torch.cat(vss_per_var_list, dim=0)  # type: ignore
    vss_complete = torch.cat(vss_complete_list, dim=0)  # type: ignore

  0%|          | 0/3620 [00:00<?, ?it/s]

100%|██████████| 3620/3620 [00:22<00:00, 163.08it/s]


In [ ]:
print(f"Mean crps for t2m and 10m wind speed: {reduce(crpss, 't f lat lon -> f', 'mean')}")
print(f"Mean es for t2m and 10m wind speed: {ess_per_var.mean(dim=0)}")
print(f"Mean es complete: {ess_complete.mean(dim=0)}")
if not SKIP_VARIOGRAM:
    print(f"Mean vs for t2m and 10m wind speed: {vss_per_var.mean(dim=0)}")
    print(f"Mean vs complete: {vss_complete.mean(dim=0)}")

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=datamodule.y_select_variables,
    scores=reduce(crpss, "t f lat lon -> f", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnsembleCRPS",
    variables=["combined"],
    scores=reduce(crpss, "t f lat lon -> 1", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=ess_per_var.mean(dim=0),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+ECC",
    metric="EnergyScore",
    variables=["combined"],
    scores=ess_complete.mean(dim=0, keepdim=True),
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+ECC",
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=vss_per_var.mean(dim=0),
    )

    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+ECC",
        metric="VariogramScore",
        variables=["combined"],
        scores=vss_complete.mean(dim=0, keepdim=True),
    )

ecc_scores = {
    "ECC": {
        "CRPS_combined": reduce(crpss, "t f lat lon -> 1", "mean").item(),
        "CRPS_2m_temperature": reduce(crpss, "t f lat lon -> f", "mean")[0].item(),
        "CRPS_10m_windspeed": reduce(crpss, "t f lat lon -> f", "mean")[1].item(),
        "EnergyScore_combined": ess_complete.mean(dim=0).item(),
        "EnergyScore_2m_temperature": ess_per_var.mean(dim=0)[0].item(),
        "EnergyScore_10m_windspeed": ess_per_var.mean(dim=0)[1].item(),
    }
}
if not SKIP_VARIOGRAM:
    ecc_scores["ECC"].update(
        {
            "VariogramScore_combined": vss_complete.mean(dim=0).item(),
            "VariogramScore_2m_temperature": vss_per_var.mean(dim=0)[0].item(),
            "VariogramScore_10m_windspeed": vss_per_var.mean(dim=0)[1].item(),
        }
    )

Mean crps for t2m and 10m wind speed: tensor([2.9691, 1.4044])
Mean es for t2m and 10m wind speed: tensor([114.1679,  59.3292])
Mean es complete: 133.0304718017578


In [ ]:
# Compute scores per lead time
def compute_scores_per_leadtime(
    prediction_timedeltas,
    crpss,
    ess_per_var,
    ess_complete,
    vss_per_var=None,
    vss_complete=None,
    method="ECC",
):
    """
    Compute scores per lead time for a given method.

    Parameters
    ----------
    prediction_timedeltas : array-like
        Array of prediction timedeltas
    crpss : torch.Tensor
        CRPS scores with shape (time, feature, lon, lat)
    ess_per_var : torch.Tensor
        Energy scores per variable with shape (time, feature)
    ess_complete : torch.Tensor
        Energy scores complete with shape (time,)
    vss_per_var : torch.Tensor, optional
        Variogram scores per variable with shape (time, feature)
    vss_complete : torch.Tensor, optional
        Variogram scores complete with shape (time,)
    method : str, default="ECC"
        Method name for the scores

    Returns
    -------
    dict
        Dictionary with scores per lead time
    """
    td = np.unique(prediction_timedeltas)
    td_str = [f"{td / np.timedelta64(1, 'h'):.0f}h" for td in td]

    scores_delta = {
        method: {
            "CRPS_combined": {},
            "CRPS_2m_temperature": {},
            "CRPS_10m_windspeed": {},
            "EnergyScore_combined": {},
            "EnergyScore_2m_temperature": {},
            "EnergyScore_10m_windspeed": {},
        }
    }

    if vss_per_var is not None and vss_complete is not None:
        scores_delta[method].update(
            {
                "VariogramScore_combined": {},
                "VariogramScore_2m_temperature": {},
                "VariogramScore_10m_windspeed": {},
            }
        )

    for delta, delta_str in zip(td, td_str):
        mask = prediction_timedeltas == delta
        print(f"Processing leadtime {delta_str} with {np.sum(mask)} samples")
        crpss_delta = crpss[mask]
        ess_per_var_delta = ess_per_var[mask]
        ess_complete_delta = ess_complete[mask]

        scores_delta[method]["CRPS_combined"][delta_str] = reduce(
            crpss_delta, "t f lat lon -> 1", "mean"
        ).item()
        scores_delta[method]["CRPS_2m_temperature"][delta_str] = reduce(
            crpss_delta, "t f lat lon -> f", "mean"
        )[0].item()
        scores_delta[method]["CRPS_10m_windspeed"][delta_str] = reduce(
            crpss_delta, "t f lat lon -> f", "mean"
        )[1].item()
        scores_delta[method]["EnergyScore_combined"][delta_str] = ess_complete_delta.mean(
            dim=0
        ).item()
        scores_delta[method]["EnergyScore_2m_temperature"][delta_str] = ess_per_var_delta.mean(
            dim=0
        )[0].item()
        scores_delta[method]["EnergyScore_10m_windspeed"][delta_str] = ess_per_var_delta.mean(
            dim=0
        )[1].item()

        if vss_per_var is not None and vss_complete is not None:
            vss_per_var_delta = vss_per_var[mask]
            vss_complete_delta = vss_complete[mask]
            scores_delta[method]["VariogramScore_combined"][delta_str] = vss_complete_delta.mean(
                dim=0
            ).item()
            scores_delta[method]["VariogramScore_2m_temperature"][delta_str] = (
                vss_per_var_delta.mean(dim=0)[0].item()
            )
            scores_delta[method]["VariogramScore_10m_windspeed"][delta_str] = (
                vss_per_var_delta.mean(dim=0)[1].item()
            )

    return scores_delta


ecc_scores_delta = compute_scores_per_leadtime(
    pred_samples_reordered.prediction_timedelta.values,
    crpss,
    ess_per_var,
    ess_complete,
    vss_per_var if not SKIP_VARIOGRAM else None,
    vss_complete if not SKIP_VARIOGRAM else None,
    method="ECC",
)
ecc_scores_delta

Processing leadtime 24h with 724 samples
Processing leadtime 48h with 724 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 724 samples
Processing leadtime 120h with 724 samples


{'ECC': {'CRPS_combined': {'24h': 2.356532573699951,
   '48h': 2.2841737270355225,
   '72h': 2.2174580097198486,
   '96h': 2.1044771671295166,
   '120h': 1.9711543321609497},
  'CRPS_2m_temperature': {'24h': 3.153785228729248,
   '48h': 3.0764048099517822,
   '72h': 3.016866683959961,
   '96h': 2.897897481918335,
   '120h': 2.700422763824463},
  'CRPS_10m_windspeed': {'24h': 1.5592809915542603,
   '48h': 1.4919419288635254,
   '72h': 1.4180470705032349,
   '96h': 1.3110578060150146,
   '120h': 1.2418861389160156},
  'EnergyScore_combined': {'24h': 143.3735809326172,
   '48h': 139.0098876953125,
   '72h': 134.96864318847656,
   '96h': 128.10366821289062,
   '120h': 119.6965560913086},
  'EnergyScore_2m_temperature': {'24h': 121.64551544189453,
   '48h': 118.47021484375,
   '72h': 115.9296646118164,
   '96h': 111.1871566772461,
   '120h': 103.60710144042969},
  'EnergyScore_10m_windspeed': {'24h': 65.98519134521484,
   '48h': 63.10420608520508,
   '72h': 59.88905334472656,
   '96h': 55.3

## GCA Postprocessing
### Step 1
Transforming to latent distribution to estimate $\Sigma$
What we need for that:
- Observations from the train Set
- Forecasts from the train set

In [67]:
train_prediction = x_ds.isel(prediction=train_idx).prediction
train_times = train_prediction.time + train_prediction.prediction_timedelta
train_times = train_times.compute().values

y_train = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(time=train_times)
    .to_dataarray("feature")
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", train_prediction.to_index()),
    )
    .swap_dims({"prediction_time": "prediction"})
)
feature_order = list(cfg.data.y_select_variables)
y_train = y_train.sel(feature=feature_order)

In [69]:
datamodule.setup(stage="fit")
predictions_train = trainer.predict(model, datamodule.train_dataloader(), return_predictions=True)
predictions_train = stack_predictions(predictions_train)
predictions_train = predictions_to_dataarray(y_train, predictions_train)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 171/171 [00:01<00:00, 142.66it/s]
(10905, 4, 37, 31)


In [71]:
def transform_to_latent_gaussian(obs_da, pred_da, eps=1e-9):
    """
    Transform observations into latent Gaussian space.

    obs_da: xarray.DataArray with dims (time, feature, longitude, latitude)
            containing observed values. Feature names must match:
            ["2m_temperature", "10m_wind_speed"]

    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray with dims (time, feature, longitude, latitude)
             containing latent Gaussian transformed observations.
    """

    prediction = obs_da.coords["prediction"]
    longitude = obs_da.coords["longitude"]
    latitude = obs_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    # ---- Temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values
    obs_temp = obs_da.sel(feature="2m_temperature").values

    # Compute CDF then transform
    cdf_temp = norm.cdf(obs_temp, loc=mu_temp, scale=sigma_temp)
    # Add clamping to avoid infs
    cdf_temp = np.clip(cdf_temp, eps, 1 - eps)
    latent_temp = norm.ppf(cdf_temp)

    # ---- Wind speed (truncated normal, [0, inf)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values
    obs_wind = obs_da.sel(feature="10m_wind_speed").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf
    cdf_wind = truncnorm.cdf(obs_wind, a=a, b=b, loc=mu_wind, scale=sigma_wind)
    # Add clamping to avoid infs
    cdf_wind = np.clip(cdf_wind, eps, 1 - eps)
    latent_wind = norm.ppf(cdf_wind)

    # ---- Stack back into xarray ----
    data = np.stack([latent_temp, latent_wind], axis=1)  # (time, feature, lon, lat)

    return xr.DataArray(
        data,
        coords={
            "prediction": prediction,
            "feature": features,
            "longitude": longitude,
            "latitude": latitude,
        },
        dims=("prediction", "feature", "longitude", "latitude"),
    )

In [72]:
latent = transform_to_latent_gaussian(y_train, predictions_train, eps=1e-7)

In [77]:
flat = latent.stack(space=("feature", "longitude", "latitude"))
# shape: (time, space)
X = flat.values
# TODO Comput sigma per lead time or overall?
# Lets go for overall for now
# But it would make sense to use diffent dependency structures for different lead times
Sigma = np.cov(X, rowvar=False)  # shape (space, space)

## Step 2: Samples from the latent Space

In [82]:
t_steps = len(predictions_xr.prediction)
n_samples = 50

latent_samples = multivariate_normal.rvs(
    mean=np.zeros(Sigma.shape[0]),
    cov=Sigma,  # type: ignore
    size=(t_steps, n_samples),  # type: ignore
)
latent_samples = latent_samples.reshape(t_steps, n_samples, *y_da.shape[1:])

## Step 3: Transform back to the initial space using the predicted Distributions

In [83]:
def inverse_transform_latent(latent_samples, pred_da, eps=1e-9):
    """
    Transform latent Gaussian samples back to original space.

    latent_samples: np.array of shape (time, n_samples, feature, lon, lat)
    pred_da: xarray.DataArray with dims (time, feature, longitude, latitude)
             features: ['2m_temperature_mu', '2m_temperature_sigma',
                        '10m_wind_speed_mu', '10m_wind_speed_sigma']

    Returns: xarray.DataArray of shape (time, sample, feature, lon, lat)
    """
    prediction = pred_da.coords["prediction"]
    lon = pred_da.coords["longitude"]
    lat = pred_da.coords["latitude"]
    features = ["2m_temperature", "10m_wind_speed"]

    n_time, n_samples, n_feature, n_lon, n_lat = latent_samples.shape

    # Transform to uniform [0,1] via standard normal CDF
    uniform_samples = norm.cdf(latent_samples)

    # Clamp the uniform samples to avoid issues with PPF
    uniform_samples = np.clip(uniform_samples, eps, 1 - eps)

    # Allocate array for original space
    orig_samples = np.zeros_like(uniform_samples)

    # ---- 2m temperature (normal) ----
    mu_temp = pred_da.sel(feature="2m_temperature_mu").values
    sigma_temp = pred_da.sel(feature="2m_temperature_sigma").values

    orig_samples[:, :, 0, :, :] = norm.ppf(
        uniform_samples[:, :, 0, :, :],
        loc=mu_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
        scale=sigma_temp[:, None, :, :],  # shape: (time, 1, lon, lat)
    )

    # ---- 10m wind speed (truncated normal, [0, ∞)) ----
    mu_wind = pred_da.sel(feature="10m_wind_speed_mu").values
    sigma_wind = pred_da.sel(feature="10m_wind_speed_sigma").values

    a = (0 - mu_wind) / sigma_wind
    b = np.inf

    orig_samples[:, :, 1, :, :] = truncnorm.ppf(
        uniform_samples[:, :, 1, :, :],
        a=a[:, None, :, :],
        b=b,
        loc=mu_wind[:, None, :, :],
        scale=sigma_wind[:, None, :, :],
    )

    return xr.DataArray(
        orig_samples,
        coords={
            "prediction": prediction,
            "sample": np.arange(n_samples),
            "feature": features,
            "longitude": lon,
            "latitude": lat,
        },
        dims=("prediction", "sample", "feature", "longitude", "latitude"),
    )


gca_preds = inverse_transform_latent(latent_samples, predictions_xr, eps=1e-7)

### Lastly compute the energy scores

In [ ]:
prediction = y_da.time
ess_per_var_list = []
vss_per_var_list = []
ess_complete_list = []
vss_complete_list = []
crpss_list = []
for t in tqdm(prediction):
    curr_obs = y_da.sel(time=t).to_numpy()
    curr_pred = gca_preds.sel(time=t).to_numpy()

    obs_t = torch.tensor(curr_obs[None, ...], dtype=torch.float32)
    pred_t = torch.tensor(curr_pred[None, ...], dtype=torch.float32)

    obs_feature = rearrange(obs_t, "b f lon lat -> b f (lon lat)")
    pred_feature = rearrange(pred_t, "b n f lon lat -> b f n (lon lat)")

    obs_complete = rearrange(obs_t, "b f lon lat -> b (f lon lat)")
    pred_complete = rearrange(pred_t, "b n f lon lat -> b n (f lon lat)")

    es_per_var = es(pred_feature, obs_feature)
    vs_per_var = vs(pred_feature, obs_feature)
    crps_per_margin = crps(pred_t, obs_t)
    es_complete = es(pred_complete, obs_complete)
    vs_complete = vs(pred_complete, obs_complete)

    ess_per_var_list.append(es_per_var)
    vss_per_var_list.append(vs_per_var)
    crpss_list.append(crps_per_margin)
    ess_complete_list.append(es_complete)
    vss_complete_list.append(vs_complete)

ess_per_var = torch.cat(ess_per_var_list, dim=0)
vss_per_var = torch.cat(vss_per_var_list, dim=0)
crpss = torch.cat(crpss_list, dim=0)
ess_complete = torch.cat(ess_complete_list, dim=0)
vss_complete = torch.cat(vss_complete_list, dim=0)

  0%|          | 0/730 [00:00<?, ?it/s]

100%|██████████| 730/730 [07:45<00:00,  1.57it/s]


In [84]:
print(f"Mean crps for t2m and 10m wind speed: {reduce(crpss, 't f lat lon -> f', 'mean')}")
print(f"Mean es for t2m and 10m wind speed: {ess_per_var.mean(dim=0)}")
print(f"Mean es complete: {ess_complete.mean(dim=0)}")
if not SKIP_VARIOGRAM:
    print(f"Mean vs for t2m and 10m wind speed: {vss_per_var.mean(dim=0)}")
    print(f"Mean vs complete: {vss_complete.mean(dim=0)}")

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=datamodule.y_select_variables,
    scores=reduce(crpss, "t f lat lon -> f", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnsembleCRPS",
    variables=["combined"],
    scores=reduce(crpss, "t f lat lon -> 1", "mean"),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnergyScore",
    variables=datamodule.y_select_variables,
    scores=ess_per_var.mean(dim=0),
)

log_scores(
    file=SCORE_FILE,
    model=f"{model_class}+GCA",
    metric="EnergyScore",
    variables=["combined"],
    scores=ess_complete.mean(dim=0, keepdim=True),
)
if not SKIP_VARIOGRAM:
    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+GCA",
        metric="VariogramScore",
        variables=datamodule.y_select_variables,
        scores=vss_per_var.mean(dim=0),
    )

    log_scores(
        file=SCORE_FILE,
        model=f"{model_class}+GCA",
        metric="VariogramScore",
        variables=["combined"],
        scores=vss_complete.mean(dim=0, keepdim=True),
    )

gca_scores = {
    "GCA": {
        "CRPS_combined": reduce(crpss, "t f lat lon -> 1", "mean").item(),
        "CRPS_2m_temperature": reduce(crpss, "t f lat lon -> f", "mean")[0].item(),
        "CRPS_10m_windspeed": reduce(crpss, "t f lat lon -> f", "mean")[1].item(),
        "EnergyScore_combined": ess_complete.mean(dim=0).item(),
        "EnergyScore_2m_temperature": ess_per_var.mean(dim=0)[0].item(),
        "EnergyScore_10m_windspeed": ess_per_var.mean(dim=0)[1].item(),
    }
}
if not SKIP_VARIOGRAM:
    gca_scores["GCA"].update(
        {
            "VariogramScore_combined": vss_complete.mean(dim=0).item(),
            "VariogramScore_2m_temperature": vss_per_var.mean(dim=0)[0].item(),
            "VariogramScore_10m_windspeed": vss_per_var.mean(dim=0)[1].item(),
        }
    )
gca_scores

Mean crps for t2m and 10m wind speed: tensor([2.9691, 1.4044])
Mean es for t2m and 10m wind speed: tensor([114.1679,  59.3292])
Mean es complete: 133.0304718017578


{'GCA': {'CRPS_combined': 2.1867592334747314,
  'CRPS_2m_temperature': 2.9690771102905273,
  'CRPS_10m_windspeed': 1.4044421911239624,
  'EnergyScore_combined': 133.0304718017578,
  'EnergyScore_2m_temperature': 114.16793823242188,
  'EnergyScore_10m_windspeed': 59.32923889160156}}

In [87]:
gca_scores_delta = compute_scores_per_leadtime(
    pred_samples_reordered.prediction_timedelta.values,
    crpss,
    ess_per_var,
    ess_complete,
    vss_per_var if not SKIP_VARIOGRAM else None,
    vss_complete if not SKIP_VARIOGRAM else None,
    method="GCA",
)
gca_scores_delta

Processing leadtime 24h with 724 samples
Processing leadtime 48h with 724 samples
Processing leadtime 72h with 724 samples
Processing leadtime 96h with 724 samples
Processing leadtime 120h with 724 samples


{'GCA': {'CRPS_combined': {'24h': 2.356532573699951,
   '48h': 2.2841737270355225,
   '72h': 2.2174580097198486,
   '96h': 2.1044771671295166,
   '120h': 1.9711543321609497},
  'CRPS_2m_temperature': {'24h': 3.153785228729248,
   '48h': 3.0764048099517822,
   '72h': 3.016866683959961,
   '96h': 2.897897481918335,
   '120h': 2.700422763824463},
  'CRPS_10m_windspeed': {'24h': 1.5592809915542603,
   '48h': 1.4919419288635254,
   '72h': 1.4180470705032349,
   '96h': 1.3110578060150146,
   '120h': 1.2418861389160156},
  'EnergyScore_combined': {'24h': 143.3735809326172,
   '48h': 139.0098876953125,
   '72h': 134.96864318847656,
   '96h': 128.10366821289062,
   '120h': 119.6965560913086},
  'EnergyScore_2m_temperature': {'24h': 121.64551544189453,
   '48h': 118.47021484375,
   '72h': 115.9296646118164,
   '96h': 111.1871566772461,
   '120h': 103.60710144042969},
  'EnergyScore_10m_windspeed': {'24h': 65.98519134521484,
   '48h': 63.10420608520508,
   '72h': 59.88905334472656,
   '96h': 55.3

In [92]:
val_scores_delta = ecc_scores_delta | gca_scores_delta
update_wandb_run(RUN_PATH, {"val": val_scores_delta})